<a href="https://colab.research.google.com/github/kevin305-dev/hello-world/blob/main/Retrieval_TextEmbeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Embeddings

> Text를 숫자로 변환하여 문장간 유사성을 비교
- 대부분 대용량 말뭉치를 통해 사전 학습된 모델(PreTrained Model)을 통해 쉽게 Embedding한다

1. 대용량 말뭉치로 임베딩 모델 훈련
2. Pre-Trained Embedding Model
3. 학습 없이 Embedding 수행 가능

> Embedding 모델
-  유료 : OpenAI, Cohere, Amazon의 text_embedding_ada_002
-  무료 : HuggingFace, bge_large_en_v1.5, Ko_sbert_nli, KoSimCSE_roberta_multitask등

In [1]:
!pip install langchain unstructured pdf2image docx2txt pdfminer

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


> Document Loaders

> Text Embeddings....1)OpenAI모델 2)HuggingFace모델
-  pdf Loading
-  tiktoken : 토큰 수 기준 Loading

In [5]:
!pip install openai langchain pypdf tiktoken

In [7]:
!pip install -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 24.8 MB/s eta 0:00:00


-  KEY 가져오기

In [10]:
import os
from google.colab import userdata

#os.environ['OPENAI_API_KEY'] = userdata.get('OPNEAI_API_KEY')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

## 1) OpenAI 모델 사용 경우

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(openai_api_key='')

# embedding_model에 embed_documents를 통해 여러개 문장을 한꺼번에 Embedding한다
# Embedding한 것들을 embeddings에 저장
embeddings = embedding_model.embed_documents(
    [
        "안녕하세요",
        "제 이름은 홍길동입니다",
        "이름이 무엇인가요?",
        "랭체인은 유용합니다.",
        "Hello World!"
    ]
)

# Embedding한 embeddings의 List의 길이 점검, 첫번째 embedding의 갯수도 점검
len(embeddings), len(embeddings[0])
# (5, 1536)
# 5 : 위의 List 문장(총 5개 문장) 하나 마다 embedding vector가 생성됨
# 1536 : 첫번째 문장에 해당하는 embedding의 갯수 = 문장을 숫자로 변환해보니 1536차원을 가진 하나의 행의 숫자로 표현됨
#        * 1536차원의 한 행으로 표현됨
# 모든 문장들이 각각 1536차원의 하나의 행으로 표현됨 => 이 embedding 값으로 유사도를 확인 가능함

-  유사도 확인을 위한 질문vs답변을 작성

In [ ]:
# embed_query를 통해 한 문장만 embedding 해보자
embedded_query_q = embedding_model.embed_query("이 대화에서 언급된 이름은 무엇입니까?")
embedded_query_a = embedding_model.embed_query("이 대화에서 언급된 이름은 홍길동입니다.")
print(len(embedded_query_q), len(embedded_query_a))
# 1536 1536

-  유사도 확인
-  가장 많이 사용하는 Cosine Similarity를 이용
-  vector 유사도를 Cosine Similarity로 표현하기 위해서 cos_sim 함수를 선언

In [9]:
from numpy import dot
from numpy.linalg import norm
import numpy as np

# Consine Similarity 함수 선언
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

-  각 문장의 vector 유사도를 확인
-  OpenAIEmbeddings 모델은 한글도 embedding 잘하는 걸 확인

In [ ]:
print(cos_sim(embedded_query_q, embedded_query_a))
print(cos_sim(embedded_query_q, embeddings[1])) # "제 이름은 홍길동입니다"
print(cos_sim(embedded_query_q, embeddings[3])) # "랭체인은 유용합니다."
# 0.9018785967795147  "이 대화에서 언급된 이름은 무엇입니까?" vs "이 대화에서 언급된 이름은 홍길동입니다."
# 0.8503907115290837  "이 대화에서 언급된 이름은 무엇입니까?" vs "제 이름은 홍길동입니다"
# 0.7762045326835891  "이 대화에서 언급된 이름은 무엇입니까?" vs "랭체인은 유용합니다."

## 2) HugginFace 'bge-small-en-v1.5' 모델 사용

-  정규화 실시 : 임베딩 벡터끼리 비교할때 정규화를 통해 유사도점검을 쉽게 하기 위함

In [3]:
!pip install -q langchain-huggingface sentence-transformers # 라이브러리 다운

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-small-en-v1.5"
model_kwargs = {"device" : "cuda"}
encode_kwargs = {"normalize_embeddings":True}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
embeddings = hf.embed_documents(
    [
        "today is monday",
        "weather is nice today",
        "what's the problem?",
        "langchain is useful",
        "Hello World!",
        "my name is morris"
    ]
)

-  bge-small-en-v1.5 모델을 통한 유사도 점검

In [13]:
BGE_query_q = hf.embed_query("Hello? who is this?")
BGE_query_a = hf.embed_query("my name is morris")

print(cos_sim(BGE_query_q, BGE_query_a))
print(cos_sim(BGE_query_q, embeddings[0]))
print(cos_sim(BGE_query_q, embeddings[5]))

0.593013822140747
0.5932040744146374
0.5930138140947216


- bge-small-en을 통한 유사도 점검
- bge-small-en > bge-small-en-v1.5 보다 좋다

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "BAAI/bge-small-en"
model_kwargs = {"device" : "cuda"}
encode_kwargs = {"normalize_embeddings":True}
hf = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

embeddings = hf.embed_documents(
    [
        "today is monday",
        "weather is nice today",
        "what's the problem?",
        "langchain is useful",
        "Hello World!",
        "my name is morris"
    ]
)

BGE_query_q = hf.embed_query("Hello? who is this?")
BGE_query_a = hf.embed_query("my name is morris")

print(cos_sim(BGE_query_q, BGE_query_a))
print(cos_sim(BGE_query_q, embeddings[0]))
print(cos_sim(BGE_query_q, embeddings[5]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

0.8402118271026476
0.8280202505454731
0.8402118108412443


-  bge-small-en 모델이 한글도 좋은지 점검

In [11]:
sentences = [
    "안녕하세요",
    "제 이름은 홍길동입니다.",
    "이름이 무엇인가요?",
    "랭체인은 유용합니다.",
    "홍길동 아버지의 이름은 홍상직입니다."
]

ko_embeddings = hf.embed_documents(sentences)

-  한국어 embedding 실력이 않좋음을 확인

In [ ]:
BGE_query_q2 = hf.embed_query("홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까?")
BGE_query_a2 = hf.embed_query("홍길동의 아버지는 엄했습니다")

print("질문: 홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까 \n", "-"*100)
print("홍길동의 아버지는 엄했습니다. \t\t 문장 유사도:", round(cos_sim(BGE_query_q2, BGE_query_a2),2))
print(sentences[1], "\t\t\t 문장 유사도:", round(cos_sim(BGE_query_q2, ko_embeddings[1]),2))
print(sentences[3], "\t\t\t 문장 유사도:", round(cos_sim(BGE_query_q2, ko_embeddings[3]),2))
print(sentences[4], "\t\t\t 문자 유사도:", round(cos_sim(BGE_query_q2, ko_embeddings[4]),2))

## 3) 한국어 사전학습 모델 임베딩 : ko-sber-nli

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "jhgan/ko-sbert-nli"
model_kwargs = {"device" : "cuda"}
encode_kwargs = {"normalize_embeddings" : True}
ko = HuggingFaceEmbeddings(
    model_name = model_name,
    model_kwargs = model_kwargs,
    encode_kwargs = encode_kwargs
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.46k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [16]:
sentences = [
    "안녕하세요",
    "제 이름은 홍길동입니다.",
    "이름이 무엇인가요?",
    "랭체인은 유용합니다.",
    "홍길동 아버지의 이름은 홍상직입니다."
]

ko_embeddings = ko.embed_documents(sentences)

- HuggingFace 'jhgan/ko-sbert-nli' 한글 Embedding 사전학습 모델의 성능이 좋다 확인

In [20]:
q = "홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까?"
a = "홍길동의 아버지는 엄했습니다"
ko_query_q = ko.embed_query(q)
ko_query_a = ko.embed_query(a)

print("질문: {} \n".format(q), "-"*100)
print("{} \t\t 문장 유사도 : ".format(a), round(cos_sim(ko_query_q, ko_query_a),2))
print("{} \t\t 문장 유사도 : ".format(sentences[1]), round(cos_sim(ko_query_q, ko_embeddings[1]),2))
print("{} \t\t\t 문장 유사도 : ".format(sentences[3]), round(cos_sim(ko_query_q, ko_embeddings[3]),2))
print("{} \t 문장 유사도 : ".format(sentences[4]), round(cos_sim(ko_query_q, ko_embeddings[4]),2))

질문: 홍길동은 아버지를 아버지라 부르지 못하였습니다. 홍길동 아버지의 이름은 무엇입니까? 
 ----------------------------------------------------------------------------------------------------
홍길동의 아버지는 엄했습니다 		 문장 유사도 :  0.46
제 이름은 홍길동입니다. 		 문장 유사도 :  0.54
랭체인은 유용합니다. 			 문장 유사도 :  0.03
홍길동 아버지의 이름은 홍상직입니다. 	 문장 유사도 :  0.61


## 4) VectorStore

## 5) Retriever